# Chapter 17 &mdash; A BDD is the Minimal DFA of a Function's On-Set

**Concept 2 of the Chapter 17 decomposition:** *A BDD is the Minimal DFA of a Function's On-Set*

Treat the on-set as a formal language over $\{0,1\}$ and minimize its DFA &mdash; that is the BDD.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-BDD-As-Minimal-DFA/Concept-BDD-As-Minimal-DFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The identification that makes this chapter part of an automata book.

A Boolean function's **on-set** is the set of input vectors where it is 1. Written as
fixed-length bit strings, that is a **finite formal language** over $\{0,1\}$.

* Build the DFA for that language.
* **Minimize** it.
* What you get *is* the **BDD**, with the variable order given by the string order.

Everything transfers: **Myhill&ndash;Nerode** gives canonicity (one minimal machine per
function, given the order), **indistinguishable states** are **identical residual
functions**, and **merging equivalent states** is exactly **sharing isomorphic
subgraphs**.

Two BDD-specific reductions: drop a node whose branches agree (the variable is
irrelevant there), and share structurally identical nodes.

## 2. Definitions

### The BDD package

In [ ]:
# --- a minimal BDD package ----------------------------------------------
# A node is either the terminal 0/1, or ('n', var_index, low, high) where
# low is the 0-branch and high the 1-branch.  Hash consing (the `unique`
# table) is what makes the representation canonical: structurally equal
# subgraphs become the SAME Python object, so equality is pointer equality.
ZERO, ONE = 0, 1

class BDD:
    def __init__(self, nvars):
        self.nvars = nvars
        self.unique = {}          # (var, low, high) -> node  -- hash consing
        self.apply_cache = {}

    def mk(self, var, low, high):
        if low is high: return low            # REDUCTION 1: skip a useless test
        key = (var, id(low), id(high), self._k(low), self._k(high))
        if key in self.unique: return self.unique[key]   # REDUCTION 2: share
        node = ('n', var, low, high)
        self.unique[key] = node
        return node

    def _k(self, n):
        return n if n in (ZERO, ONE) else ('n', n[1], self._k(n[2]), self._k(n[3]))

    def var(self, i):
        return self.mk(i, ZERO, ONE)

    def apply(self, op, a, b):
        key = (op, self._k(a), self._k(b))
        if key in self.apply_cache: return self.apply_cache[key]
        if a in (ZERO, ONE) and b in (ZERO, ONE):
            r = ONE if op(bool(a), bool(b)) else ZERO
        else:
            va = a[1] if a not in (ZERO, ONE) else self.nvars
            vb = b[1] if b not in (ZERO, ONE) else self.nvars
            v = min(va, vb)
            al, ah = (a[2], a[3]) if va == v else (a, a)
            bl, bh = (b[2], b[3]) if vb == v else (b, b)
            r = self.mk(v, self.apply(op, al, bl), self.apply(op, ah, bh))
        self.apply_cache[key] = r
        return r

    def NOT(self, a):  return self.apply(lambda x, y: not x, a, a)
    def AND(self, a, b): return self.apply(lambda x, y: x and y, a, b)
    def OR(self, a, b):  return self.apply(lambda x, y: x or y, a, b)
    def XOR(self, a, b): return self.apply(lambda x, y: x != y, a, b)

    def evaluate(self, node, assign):
        while node not in (ZERO, ONE):
            node = node[3] if assign[node[1]] else node[2]
        return bool(node)

    def size(self, node):
        seen = set()
        def walk(n):
            if n in (ZERO, ONE): return
            k = self._k(n)
            if k in seen: return
            seen.add(k); walk(n[2]); walk(n[3])
        walk(node)
        return len(seen)

    def onset(self, node, order=None):
        from itertools import product
        out = []
        for bits in product([False, True], repeat=self.nvars):
            a = {i: bits[i] for i in range(self.nvars)}
            if self.evaluate(node, a):
                out.append(''.join('1' if bits[i] else '0' for i in range(self.nvars)))
        return sorted(out)

### Building the DFA of an on-set, and minimizing it

In [ ]:
from itertools import product
def onset_dfa(onset, N):
    # a trie over fixed-length bit strings, accepting exactly `onset`
    lines = ['DFA']
    def nm(prefix):
        if prefix == '': return 'I'
        return ('F' if (len(prefix) == N and prefix in onset) else 'S') + '_' + prefix
    for k in range(N):
        for p in product('01', repeat=k):
            pre = ''.join(p)
            for b in '01':
                lines.append('%s : %s -> %s' % (nm(pre), b, nm(pre + b)))
    for p in product('01', repeat=N):
        s = ''.join(p)
        lines.append('%s : 0 | 1 -> BH' % nm(s))
    lines.append('BH : 0 | 1 -> BH')
    return md2mc('\n'.join(lines))

## 3. Tests

A function, its on-set, and the DFA of that on-set.

In [ ]:
N = 3
f = lambda bits: int(sum(bits) >= 2)              # majority
onset = {''.join(str(b) for b in bits)
         for bits in product([0, 1], repeat=N) if f(bits)}
print("on-set :", sorted(onset))
D = onset_dfa(onset, N)
print("trie DFA  : %d states" % len(D["Q"]))
Dm = min_dfa(D)
print("minimized : %d states" % len(Dm["Q"]))

The minimal DFA accepts exactly the on-set.

In [ ]:
acc = {''.join(p) for p in product('01', repeat=N) if accepts_dfa(Dm, ''.join(p))}
print("accepted :", sorted(acc))
assert acc == onset

**And the BDD has the same shape.**

In [ ]:
b = BDD(N)
x0, x1, x2 = b.var(0), b.var(1), b.var(2)
maj = b.OR(b.OR(b.AND(x0, x1), b.AND(x0, x2)), b.AND(x1, x2))
print("BDD nodes (non-terminal) :", b.size(maj))
print("BDD on-set               :", b.onset(maj))
assert set(b.onset(maj)) == onset
print("\nminimal DFA states %d (incl. terminals and sink), BDD nodes %d"
      % (len(Dm["Q"]), b.size(maj)))

**Indistinguishable states = identical residual functions.**

In [ ]:
def residual(f, N, prefix):
    return tuple(f(tuple(int(c_) for c_ in prefix) + rest)
                 for rest in product([0, 1], repeat=N - len(prefix)))
groups = {}
for k in range(N + 1):
    for p in product('01', repeat=k):
        pre = ''.join(p)
        groups.setdefault(residual(f, N, pre), []).append(pre)
print("distinct residual functions : %d" % len(groups))
for r, pres in sorted(groups.items(), key=lambda kv: len(kv[1])):
    print("   %-18s reached by %s" % (str(r), pres))

**Reduction 1:** a node whose two branches agree is dropped.

In [ ]:
b2 = BDD(3)
y0, y1 = b2.var(0), b2.var(1)
f_no_x2 = b2.AND(y0, y1)                 # x2 is irrelevant
print("nodes in AND(x0, x1) over 3 variables :", b2.size(f_no_x2))
assert b2.size(f_no_x2) == 2
print("no node tests x2 -- mk() returned `low` because low is high")

**Reduction 2:** structurally identical subgraphs are shared, not copied.

In [ ]:
b3 = BDD(4)
z = [b3.var(i) for i in range(4)]
g = b3.OR(b3.AND(z[0], z[2]), b3.AND(z[1], z[2]))
print("nodes :", b3.size(g))
print("unique table entries :", len(b3.unique))
print("\nThe two occurrences of x2 became ONE node.  That is hash consing,")
print("and it is the same act as merging two indistinguishable DFA states.")

## 4. Animation

The minimal DFA of a function's on-set &mdash; which is to say, its BDD.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(onset_dfa(onset, 3)), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Build the on-set DFA for XOR of three variables. How many states after minimizing?
2. Why does the DFA need a sink state that the BDD does not?
3. State Myhill&ndash;Nerode in BDD vocabulary.

In [ ]:
# Your work for the exercises above.